## Gold Layer — Recent Activities Enriched

Enriched view of the last 90 days of activities with full context for training coach AI recommendations.

**Purpose:**
* Contextual workout suggestions based on recent patterns
* Equipment and weather-based insights
* Day/time pattern analysis
* Recovery and effort tracking

**Source:** `garmin_lakehouse.silver.activity_details`

**Refresh Cadence:** Daily (or more frequently for real-time coaching)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.gold.recent_activities_enriched AS

SELECT
    -- Core identifiers
    ad.activity_id,
    ad.activity_name,
    ad.description,
    ad.activity_type_key,
    
    -- Timing context
    ad.start_date,
    DAYOFWEEK(ad.start_date) as day_of_week,
    CASE DAYOFWEEK(ad.start_date)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END as day_name,
    ad.start_hour,
    CASE
        WHEN ad.start_hour < 6 THEN 'Early Morning'
        WHEN ad.start_hour < 12 THEN 'Morning'
        WHEN ad.start_hour < 17 THEN 'Afternoon'
        WHEN ad.start_hour < 21 THEN 'Evening'
        ELSE 'Night'
    END as time_of_day_label,
    
    -- Performance metrics
    ad.distance_km,
    ad.duration_minutes,
    ad.avg_pace_min_per_km,
    ad.avg_pace_formatted,
    ad.avg_heart_rate,
    ad.max_heart_rate,
    ad.avg_cadence_spm,
    ad.avg_stride_length_m,
    
    -- Training load
    ad.aerobic_training_effect,
    ad.anaerobic_training_effect,
    ad.vo2_max,
    
    -- Calculated effort score (0-10 scale based on HR zones and training effects)
    ROUND(
        COALESCE(
            (ad.hr_zone_3_seconds * 0.3 + 
             ad.hr_zone_4_seconds * 0.6 + 
             ad.hr_zone_5_seconds * 1.0) / 
            NULLIF(ad.duration_minutes * 60, 0) * 10,
            (ad.aerobic_training_effect + ad.anaerobic_training_effect) / 2 * 2
        ), 
    1) as effort_score,
    
    -- Elevation
    ad.elevation_gain_m,
    ad.elevation_loss_m,
    
    -- Gear context
    ad.gear_name,
    ad.gear_type,
    ad.has_gear,
    
    -- Weather context
    ad.temperature_c,
    ad.feels_like_c,
    ad.wind_speed_kmh,
    ad.weather_condition,
    ad.has_weather,
    
    -- Location
    ad.location_name,
    ad.start_latitude,
    ad.start_longitude,
    
    -- Recovery indicator (days since this activity)
    DATEDIFF(CURRENT_DATE(), ad.start_date) as days_ago,
    
    -- Calories
    ad.total_calories,
    
    -- Metadata
    ad.loaded_at
    
FROM garmin_lakehouse.silver.activity_details ad
WHERE ad.start_date >= DATE_SUB(CURRENT_DATE(), 90)  -- Last 90 days
ORDER BY ad.start_date DESC

In [0]:
%sql
SELECT
    activity_name,
    start_date,
    day_name,
    time_of_day_label,
    activity_type_key,
    distance_km,
    duration_minutes,
    avg_pace_formatted,
    effort_score,
    temperature_c,
    weather_condition,
    gear_name
FROM garmin_lakehouse.gold.recent_activities_enriched
ORDER BY start_date DESC
LIMIT 20